In [1]:
import os
%pwd
os.chdir("../")
%pwd

'/home/aditya/Desktop/MLOPS/FinancialPhraseClassifier'

In [2]:
from dataclasses import dataclass
from pathlib import Path
from transformers import AutoTokenizer
from datasets import load_from_disk
from src.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.utils.common import read_yaml
from box import ConfigBox

/home/aditya/Desktop/MLOPS/FinancialPhraseClassifier/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
@dataclass()
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str

In [6]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        os.makedirs(self.config.artifacts_root, exist_ok=True)

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        os.makedirs(config.root_dir, exist_ok=True)

        return DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            tokenizer_name=config.tokenizer_name
        )

In [7]:
config_mgr = ConfigurationManager()
trans_config = config_mgr.get_data_transformation_config()

tokenizer = AutoTokenizer.from_pretrained(trans_config.tokenizer_name)
dataset = load_from_disk(trans_config.data_path)

In [8]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)
output_path = os.path.join(trans_config.root_dir, "finbert_dataset")
tokenized_dataset.save_to_disk(output_path)

print("Columns generated:", tokenized_dataset["train"].column_names)
print("Saved successfully to:", output_path)

Saving the dataset (1/1 shards): 100%|██████████| 2388/2388 [00:00<00:00, 496702.11 examples/s]

Columns generated: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']
Saved successfully to: artifacts/data_transformation/finbert_dataset
